# Derivation: The Daily Update of the Single Index Model
The [lecture](../../CHEME-5660-L15a-Lecture-Testing-Trading-Strategies-Fall-2026.ipynb) updates the baseline and market sensitivity of each stock every trading day with exponentially weighted least squares (EWLS). How can we solve the EWLS problem each day without storing every past day?

> __Learning Objectives:__
>
> By the end of this derivation, you should be able to:
> * __Write the problem as one weighted sum:__ Combine the new days and the 2016 to 2024 days into one weighted sum of squared residuals. Solve the weighted normal equations for the baseline and the market sensitivity, and compute the residual standard deviation.
> * __Update the normal equations each day:__ Multiply the weighted sums by the decay and add the new day, without storing any past day.
> * __Start from the 2016 to 2024 fit:__ Compute the starting values from the fixed estimates, so that the update starts at the 2016 to 2024 fit.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L15a) lists these symbols with their meanings and units.

Let's get started!
___

## The Estimates from One Weighted Sum
The lecture numbers the trading days from the end of 2024. Day $0$ is December 31, 2024, days $-2262$ to $0$ are the 2,263 days of 2016 to 2024, and day $t$ is the day of the update. On day $t$, the baseline $\alpha_{i}(t)$ and market sensitivity $\beta_{i}(t)$ of stock $i$ solve:

$$
\left(\alpha_{i}(t),\beta_{i}(t)\right) = \arg\min_{\alpha_{i},\,\beta_{i}}\Biggl[\underbrace{\sum_{k=1}^{t}\phi^{\,t-k}\,r_{i}(k)^{2}}_{\text{new days}} + \underbrace{\phi^{\,t}\,\frac{N_{0}}{2263}\sum_{k=-2262}^{0}r_{i}(k)^{2}}_{\text{prior}}\Biggr]
$$

where $r_{i}(k)=g_{i}(k)-\alpha_{i}-\beta_{i}\,g_{m}(k)$ is the residual of stock $i$ on day $k$ for a candidate baseline $\alpha_{i}$ and market sensitivity $\beta_{i}$.

The two sums differ only in their weights. Let $\omega_{t}(k)$ be the weight of day $k$ on day $t$. The problem becomes one weighted sum over days $-2262$ to $t$:

$$
\omega_{t}(k) = \begin{cases}\phi^{\,t-k} & \text{new days, } k=1,\dots,t\\[4pt] \dfrac{N_{0}}{2263}\,\phi^{\,t} & \text{prior, } k=-2262,\dots,0\end{cases}
\qquad\qquad
\left(\alpha_{i}(t),\beta_{i}(t)\right) = \arg\min_{\alpha_{i},\,\beta_{i}}\sum_{k=-2262}^{t}\omega_{t}(k)\,r_{i}(k)^{2}
$$

The problem is the least-squares fit of the single index model, with a weight on each day. With equal weights, the fit gives the fixed estimates $\hat{\alpha}_{i}$ and $\hat{\beta}_{i}$.

Previously, we solved the equal-weight fit with the normal equations. Let $\mathbf{x}(k)=[1,\;g_{m}(k)]^{\top}$ be the regressor vector of day $k$ and $\boldsymbol{\theta}=[\alpha_{i},\;\beta_{i}]^{\top}$ the candidate parameters, so that $r_{i}(k)=g_{i}(k)-\mathbf{x}(k)^{\top}\boldsymbol{\theta}$.

Setting the gradient of the weighted sum to zero gives the weighted normal equations, where sums without limits run over days $-2262$ to $t$:

$$
\begin{align*}
\nabla_{\boldsymbol{\theta}}\sum_{k}\omega_{t}(k)\,r_{i}(k)^{2} &= -2\sum_{k}\omega_{t}(k)\,\mathbf{x}(k)\left(g_{i}(k)-\mathbf{x}(k)^{\top}\boldsymbol{\theta}\right) = \mathbf{0} && \text{gradient of the weighted sum}\\
\underbrace{\left(\sum_{k}\omega_{t}(k)\,\mathbf{x}(k)\,\mathbf{x}(k)^{\top}\right)}_{\mathbf{A}(t)}\boldsymbol{\theta} &= \underbrace{\sum_{k}\omega_{t}(k)\,\mathbf{x}(k)\,g_{i}(k)}_{\mathbf{c}(t)} && \text{divide by } -2\text{, move the } \boldsymbol{\theta} \text{ term to the left}\\
\boldsymbol{\theta}_{i}(t) = \begin{bmatrix}\alpha_{i}(t)\\ \beta_{i}(t)\end{bmatrix} &= \boxed{\mathbf{A}(t)^{-1}\,\mathbf{c}(t)} && \text{solve the } 2\times 2 \text{ system}
\end{align*}
$$

Written out, $\mathbf{A}(t)$ and $\mathbf{c}(t)$ are weighted sums of the growth rates:

$$
\mathbf{A}(t) = \sum_{k}\omega_{t}(k)\begin{bmatrix}1 & g_{m}(k)\\ g_{m}(k) & g_{m}(k)^{2}\end{bmatrix}, \qquad \mathbf{c}(t) = \sum_{k}\omega_{t}(k)\begin{bmatrix}g_{i}(k)\\ g_{m}(k)\,g_{i}(k)\end{bmatrix}
$$

The top-left entry of $\mathbf{A}(t)$ is the total weight $S(t)=\sum_{k}\omega_{t}(k)$.

__The residual standard deviation:__ The covariance of the growth rates also needs the residual standard deviation $\sigma_{\varepsilon,i}(t)$ of each stock. The square of $\sigma_{\varepsilon,i}(t)$ is the weighted average of the squared residuals at the estimates. Computing $\sigma_{\varepsilon,i}(t)$ takes one more weighted sum, $d(t)=\sum_{k}\omega_{t}(k)\,g_{i}(k)^{2}$:

$$
\begin{align*}
\sigma_{\varepsilon,i}(t)^{2} &= \frac{1}{S(t)}\sum_{k}\omega_{t}(k)\left(g_{i}(k)-\mathbf{x}(k)^{\top}\boldsymbol{\theta}_{i}(t)\right)^{2} && \text{weighted average squared residual}\\
&= \frac{1}{S(t)}\left(d(t)-2\,\boldsymbol{\theta}_{i}(t)^{\top}\mathbf{c}(t)+\boldsymbol{\theta}_{i}(t)^{\top}\mathbf{A}(t)\,\boldsymbol{\theta}_{i}(t)\right) && \text{expand the square, sum each term}\\
&= \frac{1}{S(t)}\left(d(t)-\boldsymbol{\theta}_{i}(t)^{\top}\mathbf{c}(t)\right) && \mathbf{A}(t)\,\boldsymbol{\theta}_{i}(t)=\mathbf{c}(t)\\
\sigma_{\varepsilon,i}(t) &= \boxed{\sqrt{\frac{d(t)-\boldsymbol{\theta}_{i}(t)^{\top}\mathbf{c}(t)}{S(t)}}} && \text{take the square root}
\end{align*}
$$

___

## The Daily Update
Recomputing $\mathbf{A}(t)$, $\mathbf{c}(t)$, and $d(t)$ from scratch each day means storing every past day. Instead, we update all three from their values on the day before. From day $t-1$ to day $t$, every earlier weight shrinks by the factor $\phi$, and the new day gets weight one:

$$
\begin{align*}
\omega_{t}(k) &= \phi^{\,t-k} = \phi\cdot\phi^{\,(t-1)-k} = \phi\,\omega_{t-1}(k) && \text{new days, } k=1,\dots,t-1\\
\omega_{t}(k) &= \frac{N_{0}}{2263}\,\phi^{\,t} = \phi\cdot\frac{N_{0}}{2263}\,\phi^{\,t-1} = \phi\,\omega_{t-1}(k) && \text{prior, } k=-2262,\dots,0\\
\omega_{t}(t) &= \phi^{\,0} = 1 && \text{the new day}
\end{align*}
$$

To update $\mathbf{A}(t)$, we split its sum into the earlier days and the new day:

$$
\begin{align*}
\mathbf{A}(t) &= \sum_{k=-2262}^{t-1}\omega_{t}(k)\,\mathbf{x}(k)\,\mathbf{x}(k)^{\top} + \omega_{t}(t)\,\mathbf{x}(t)\,\mathbf{x}(t)^{\top} && \text{earlier days and the new day}\\
&= \phi\sum_{k=-2262}^{t-1}\omega_{t-1}(k)\,\mathbf{x}(k)\,\mathbf{x}(k)^{\top} + \mathbf{x}(t)\,\mathbf{x}(t)^{\top} && \omega_{t}(k)=\phi\,\omega_{t-1}(k)\text{ and }\omega_{t}(t)=1\\
&= \phi\,\mathbf{A}(t-1) + \mathbf{x}(t)\,\mathbf{x}(t)^{\top} && \text{definition of } \mathbf{A}(t-1)
\end{align*}
$$

The same steps update $\mathbf{c}(t)$ and $d(t)$:

$$
\boxed{\mathbf{A}(t) = \phi\,\mathbf{A}(t-1) + \mathbf{x}(t)\,\mathbf{x}(t)^{\top}, \qquad \mathbf{c}(t) = \phi\,\mathbf{c}(t-1) + \mathbf{x}(t)\,g_{i}(t), \qquad d(t) = \phi\,d(t-1) + g_{i}(t)^{2}}
$$

Each day, the update multiplies the three sums by $\phi$, adds the new day, and solves the $2\times 2$ system for $\alpha_{i}(t)$ and $\beta_{i}(t)$. We never store a past day. Each update takes the same small amount of work.

The decay $\phi=2^{-1/H}$ halves the weight of a day every $H$ trading days. The trading method uses a half-life of $H=63$ days, one trading quarter. A shorter half-life follows changes in the stock faster but gives noisier estimates, while a longer half-life gives smoother estimates that follow changes more slowly.

___

## Starting Values from the 2016 to 2024 Fit
The update starts on day $0$, December 31, 2024. On that day, the sums hold only the 2,263 days of 2016 to 2024, each with weight $N_{0}/2263$. A bar marks an average over those days, such as $\overline{g_{m}}$ for the average of $g_{m}(k)$.

The fixed estimates $\hat{\boldsymbol{\theta}}_{i}=[\hat{\alpha}_{i},\;\hat{\beta}_{i}]^{\top}$ solve the equal-weight normal equations over days $-2262$ to $0$, $\left(\sum\mathbf{x}(k)\,\mathbf{x}(k)^{\top}\right)\hat{\boldsymbol{\theta}}_{i}=\sum\mathbf{x}(k)\,g_{i}(k)$. Multiplying both sides by $N_{0}/2263$ gives the starting values:

$$
\begin{align*}
\mathbf{A}(0) &= \frac{N_{0}}{2263}\sum_{k=-2262}^{0}\mathbf{x}(k)\,\mathbf{x}(k)^{\top} = N_{0}\begin{bmatrix}1 & \overline{g_{m}}\\ \overline{g_{m}} & \overline{g_{m}^{2}}\end{bmatrix} && \text{2,263 days, each with weight } N_{0}/2263\\
\mathbf{c}(0) &= \frac{N_{0}}{2263}\sum_{k=-2262}^{0}\mathbf{x}(k)\,g_{i}(k) = \mathbf{A}(0)\,\hat{\boldsymbol{\theta}}_{i} && \text{normal equations of the fit, times } N_{0}/2263\\
\boldsymbol{\theta}_{i}(0) &= \mathbf{A}(0)^{-1}\,\mathbf{c}(0) = \hat{\boldsymbol{\theta}}_{i} && \text{the update starts at the fit}
\end{align*}
$$

The starting values need only the fixed estimates and the averages $\overline{g_{m}}$ and $\overline{g_{m}^{2}}$. We choose $d(0)$ so that the update also starts at the residual standard deviation $\hat{\sigma}_{\varepsilon,i}$ of the 2016 to 2024 fit:

$$
\begin{align*}
d(0) &= \hat{\boldsymbol{\theta}}_{i}^{\top}\,\mathbf{c}(0) + N_{0}\,\hat{\sigma}_{\varepsilon,i}^{2} && \text{starting value}\\
\sigma_{\varepsilon,i}(0)^{2} &= \frac{d(0)-\hat{\boldsymbol{\theta}}_{i}^{\top}\,\mathbf{c}(0)}{S(0)} = \frac{N_{0}\,\hat{\sigma}_{\varepsilon,i}^{2}}{N_{0}} = \hat{\sigma}_{\varepsilon,i}^{2} && \text{boxed formula, with } S(0)=N_{0}
\end{align*}
$$

A large prior weight $N_{0}$ keeps the estimates near the 2016 to 2024 fit for longer. A small $N_{0}$ lets the first new days move the estimates quickly. The trading method uses $N_{0}=63$.

___

## Summary
We solved the lecture's EWLS problem with weighted normal equations that update each day.

> __Key Takeaways:__
>
> * __One weighted sum:__ The new days and the 2016 to 2024 days enter one sum of squared residuals with different weights. Solving the weighted normal equations gives the baseline and the market sensitivity, as in ordinary least squares.
> * __One step per day:__ Each day, every earlier weight shrinks by the decay, and the new day enters with weight one. Each weighted sum then updates in one step, without storing any past day.
> * __Start at the fixed estimates:__ The starting values need only the fixed estimates and two averages over 2016 to 2024. On December 31, 2024, the update returns the 2016 to 2024 fit.

Return to the [lecture](../../CHEME-5660-L15a-Lecture-Testing-Trading-Strategies-Fall-2026.ipynb), or explore the [example](../../CHEME-5660-L15a-Example-Updating-SIM-Gamma-Fall-2026.ipynb), which updates the estimates of every stock in our universe.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___